# Explaining Forest Cover Type Predictions with LIME

## Aim

This notebook trains an XGBoost classifier to predict forest cover type from cartographic and environmental measurements, evaluates it on held-out data, and uses **LIME Tabular** to explain individual predictions. It uses one XAI method throughout: LIME’s tabular classifier explainer.

The original Covertype dataset has 581,012 observations, 54 input features, and 7 target classes. To keep training and explanation runs practical on a laptop, the modeling workflow takes a reproducible, stratified sample of 10,000 records from the full dataset.


## 1. Imports and reproducibility

The shared protocol fixes the data sample, train/test split, explanation cases, LIME background, feature subsets, perturbation budgets, and repeat seeds. This notebook keeps XGBoost's model configuration and model fitting specific to XGBoost. The first data-loading run may fetch Covertype through scikit-learn if it is not already cached.


In [ ]:
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)
from xgboost import XGBClassifier
from lime.lime_tabular import LimeTabularExplainer

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (9, 5)
%matplotlib inline


## 2. Dataset and prediction objective

The **Covertype** dataset describes forest areas in the Roosevelt National Forest of northern Colorado. Each row represents a 30 × 30 metre cell. The objective is to predict its forest cover type from the cartographic variables. The dataset was assembled for predicting cover type from variables available from a US Geological Survey (USGS) and US Forest Service (USFS) digital map.

### Target classes

The original labels 1–7 identify: Spruce/Fir, Lodgepole Pine, Ponderosa Pine, Cottonwood/Willow, Aspen, Douglas-fir, and Krummholz. The code remaps them to 0–6 for XGBoost while retaining these names in reports and plots.

### Input features

There are 54 features:

| Feature group | Features | What they represent |
|---|---|---|
| Elevation | `Elevation` | Elevation in metres |
| Terrain | `Aspect`, `Slope` | Aspect in degrees azimuth and slope in degrees |
| Hydrology | `Horizontal_Distance_To_Hydrology`, `Vertical_Distance_To_Hydrology` | Horizontal and vertical distance to surface water |
| Access / fire | `Horizontal_Distance_To_Roadways`, `Horizontal_Distance_To_Fire_Points` | Horizontal distance to roads and fire ignition points |
| Hillshade | `Hillshade_9am`, `Hillshade_Noon`, `Hillshade_3pm` | Modelled hillshade at 9 a.m., noon, and 3 p.m. (0–255) |
| Wilderness area | `Wilderness_Area_0`–`Wilderness_Area_3` | Four binary indicators for wilderness areas |
| Soil type | `Soil_Type_0`–`Soil_Type_39` | Forty binary indicators for soil types |

The four wilderness indicators and forty soil indicators are one-hot / binary features. Distances and elevation are measured in metres; aspect and slope are angular measurements.

### Dataset source

The notebook loads Covertype with `sklearn.datasets.fetch_covtype()`. Scikit-learn uses its own dataset cache and fetches the data automatically if it is not cached; no project-local dataset archive is required.


In [ ]:
import sys
from pathlib import Path

for _directory in (Path.cwd(), Path.cwd() / "experiments", Path.cwd() / "XAI" / "experiments"):
    if (_directory / "controlled_lime_protocol.py").is_file():
        sys.path.insert(0, str(_directory))
        break
else:
    raise FileNotFoundError("Keep controlled_lime_protocol.py beside the experiment notebooks.")

from controlled_lime_protocol import (
    CLASS_NAMES, RANDOM_STATE, SAMPLE_SIZE, TEST_SIZE, BACKGROUND_SIZE,
    SAMPLE_BUDGETS, REPEATS, EXPLANATIONS_PER_CLASS, NUM_FEATURES, TOP_K,
    FEATURE_COUNTS, load_controlled_covtype, selected_explanation_cases,
    benchmark_lime, evaluate_predictive_performance,
)

X, y, X_train, X_test, y_train, y_test = load_controlled_covtype()
feature_names = X.columns.tolist()
class_names = list(CLASS_NAMES)
lime_background = X_train.sample(n=min(BACKGROUND_SIZE, len(X_train)), random_state=RANDOM_STATE)
explanation_cases = selected_explanation_cases(X_test, y_test)

print(f"Shared modeling sample: {len(X):,} rows, {X.shape[1]} features")
print(f"Train/test: {len(X_train):,} / {len(X_test):,} rows; seed: {RANDOM_STATE}")
print(f"LIME background: {len(lime_background):,} rows")
print(f"Cases: {len(explanation_cases)} ({EXPLANATIONS_PER_CLASS} per true class)")
print("Case indices:", explanation_cases.index.tolist())
print("Perturbation dimensions:", FEATURE_COUNTS)
print("Budgets:", SAMPLE_BUDGETS, "Repeats:", REPEATS, "Explanation features:", NUM_FEATURES)
X.head()


### Class counts in the modeling sample

The full dataset is imbalanced: some forest types occur much more often than others. This plot shows class counts in our 10,000-row stratified sample. We also stratify the train/test split so each subset retains similar class proportions.


In [ ]:
class_counts = y.map(lambda class_id: class_names[class_id]).value_counts().reindex(class_names)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_counts.index, class_counts.values, color=plt.cm.tab10(np.arange(len(class_names))))
ax.set(title="Cover type counts in the stratified modeling sample", xlabel="Cover type", ylabel="Number of cells")
ax.tick_params(axis="x", labelrotation=25)
for bar, value in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 100, f"{value:,}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()


### Continuous feature distributions

The first ten features are continuous measurements and use different scales. We standardize them **for this plot only** so the distributions can be compared; XGBoost is trained on the original values. Boxplots show how these measurements vary across cover types.


In [ ]:
continuous_features = feature_names[:10]
X_cont = X[continuous_features]
z = (X_cont - X_cont.mean()) / X_cont.std(ddof=0)
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
class_colors = plt.cm.tab10(np.arange(len(class_names)))
for ax, feature in zip(axes.flat, continuous_features):
    grouped = [z.loc[y == class_id, feature].to_numpy() for class_id in range(len(class_names))]
    bp = ax.boxplot(grouped, patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], class_colors):
        patch.set_facecolor(color)
    ax.set_title(feature.replace("_", " "), fontsize=9)
    ax.set_xticks(range(1, len(class_names) + 1), labels=range(1, len(class_names) + 1))
    ax.set_xlabel("Cover type code", fontsize=8)
    ax.set_ylabel("z-score", fontsize=8)
fig.suptitle("Continuous feature distributions by cover type (standardized for display)", y=1.02)
plt.tight_layout()
plt.show()


### Wilderness and soil indicator prevalence

This heatmap shows the share of observations in each cover type with each binary area/soil indicator set to 1. It helps show which soil and wilderness indicators are more common within each class. These are associations in this dataset, not causal effects.


In [ ]:
indicator_features = feature_names[10:]
prevalence = X.assign(cover_type=y).groupby("cover_type")[indicator_features].mean().reindex(range(len(class_names)))
fig, ax = plt.subplots(figsize=(18, 5))
image = ax.imshow(prevalence.to_numpy(), aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
ax.set_yticks(range(len(class_names)), labels=class_names)
short_names = [f"W{i}" for i in range(1, 5)] + [f"S{i}" for i in range(1, 41)]
ax.set_xticks(range(len(indicator_features)), labels=short_names, rotation=90, fontsize=8)
ax.set_title("Share of cells with each wilderness (W) and soil (S) indicator")
fig.colorbar(image, ax=ax, label="Share of class with indicator = 1", shrink=0.8)
plt.tight_layout()
plt.show()


### Pairwise Pearson correlations between continuous features

The heatmap compares the ten continuous measurements pair by pair. Pearson correlation ranges from −1 to +1 and describes linear association. Correlation does not establish causation; the binary soil and wilderness indicators are left out of this particular plot.


In [ ]:
corr = X[continuous_features].corr()
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=70, ha="right")
ax.set_yticks(range(len(corr.index)), labels=corr.index)
fig.colorbar(image, ax=ax, label="Pearson correlation", shrink=0.8)
ax.set_title("Correlations between continuous terrain features")
plt.tight_layout()
plt.show()


## 3. Train XGBoost on the shared split

`load_controlled_covtype()` supplies the seeded, stratified 10,000-row sample and its 80/20 train/test split. We use those exact training and test rows here, so the XGBoost run is directly comparable on data with the other model notebooks using the protocol. The model settings and fitting remain specific to XGBoost. The test set is not used to fit the model.


In [ ]:
# X_train, X_test, y_train, and y_test come from load_controlled_covtype()
# in the dataset setup cell. Reuse that split for training and evaluation.
model = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=4,
)
model.fit(X_train, y_train)

print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print("Training class proportions:")
print(y_train.value_counts(normalize=True).sort_index().rename(index=dict(enumerate(class_names))).round(3))
print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).sort_index().rename(index=dict(enumerate(class_names))).round(3))


## 4. Check XGBoost general performance

Five-fold cross-validation estimates consistency across several train/validation partitions of the training split. We then report metrics on the untouched test split. Because the classes are imbalanced, inspect precision, recall, and F1 for each class in addition to accuracy. Macro F1 gives each cover type equal weight; weighted F1 reflects class frequency.


In [ ]:
model_params = dict(
    n_estimators=250, max_depth=6, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    objective="multi:softprob", eval_metric="mlogloss",
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=4,
)
cv_scores = cross_val_score(
    XGBClassifier(**model_params), X_train, y_train, cv=5, scoring="accuracy", n_jobs=1
)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(f"5-fold training CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Held-out test accuracy:      {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, labels=np.arange(len(class_names)), target_names=class_names, digits=3, zero_division=0))


### Confusion matrix

Rows show the true cover type and columns show predictions. The diagonal contains correct classifications. Off-diagonal cells show which forest types the model confused.


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(10, 8))
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
cm_display.plot(ax=ax, cmap="Blues", values_format=",d", colorbar=False, xticks_rotation=35)
plt.title("Held-out test confusion matrix")
plt.tight_layout()
plt.show()


### Per-class precision, recall, and F1

This grouped bar chart compares class-wise precision, recall, and F1 on the same held-out predictions. A class with lower recall is often being missed; lower precision means predictions of that class are more often wrong.


In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=np.arange(len(class_names)), zero_division=0
)
metrics = pd.DataFrame({"Precision": precision, "Recall": recall, "F1": f1}, index=class_names)
fig, ax = plt.subplots(figsize=(13, 6))
positions = np.arange(len(class_names))
width = 0.24
for offset, (metric_name, color) in enumerate(zip(metrics.columns, ["#4C78A8", "#F58518", "#54A24B"])):
    ax.bar(positions + (offset - 1) * width, metrics[metric_name], width, label=metric_name, color=color)
ax.set(title="Per-class test performance", xlabel="Cover type", ylabel="Score", ylim=(0, 1.05))
ax.set_xticks(positions, labels=class_names, rotation=25, ha="right")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()
display(metrics.round(3))


## 5. Illustrative LIME explanation

Use the same background, first shared case, and explanation settings as the other notebooks.


In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# Use the protocol's background and categorical handling for the illustration too.
categorical_feature_indices = list(range(10, X.shape[1]))
explainer = LimeTabularExplainer(
    training_data=lime_background.to_numpy(),
    feature_names=feature_names,
    class_names=class_names,
    categorical_features=categorical_feature_indices,
    categorical_names={i: ["0", "1"] for i in categorical_feature_indices},
    mode="classification",
    random_state=RANDOM_STATE,
)

def predict_proba_for_lime(rows):
    return model.predict_proba(pd.DataFrame(rows, columns=feature_names))


### First shared held-out case

Case selection does not depend on model confidence or correctness.


In [ ]:
# Select the same protocol case in every notebook, independently of predictions.
case_index = explanation_cases.index[0]
instance = X_test.loc[case_index]
true_class = int(y_test.loc[case_index])
predicted_class = int(model.predict(X_test.loc[[case_index]])[0])
predicted_class_index = int(np.flatnonzero(model.classes_ == predicted_class)[0])
probabilities = predict_proba_for_lime(instance.to_numpy()[None, :])[0]
print("Shared test row:", case_index)
print("Actual:", class_names[true_class], "| Predicted:", class_names[predicted_class])

# This single explanation illustrates the result; the comparison uses all shared cases.
explanation = explainer.explain_instance(
    data_row=instance.to_numpy(),
    predict_fn=predict_proba_for_lime,
    labels=(predicted_class_index,),
    num_features=NUM_FEATURES,
    num_samples=SAMPLE_BUDGETS[-2],
)

display(pd.Series(probabilities, index=class_names).to_frame("probability").round(3))
display(pd.DataFrame(explanation.as_list(label=predicted_class_index),
    columns=["feature condition", "LIME weight"]).round(3))


### Local feature contributions

Positive weights support the selected class in LIME’s local surrogate; negative weights count against it. Continuous-feature conditions show value ranges, while soil and wilderness features may appear as binary conditions. The weights are not changes in XGBoost’s class probability. The displayed example uses the first run at the largest sample budget from the benchmark above.


In [ ]:
local_terms = explanation.as_list(label=predicted_class_index)
local_df = pd.DataFrame(local_terms, columns=["condition", "weight"]).sort_values("weight")
colors = ["#4C78A8" if value < 0 else "#F58518" for value in local_df.weight]
plt.figure(figsize=(10, 6))
plt.barh(local_df.condition, local_df.weight, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("LIME surrogate weight for predicted cover type")
plt.title(f"Local LIME explanation: {class_names[predicted_class]}")
plt.tight_layout()
plt.show()


### Illustrations for one fixed shared case per true class

These plots use protocol cases and settings; they are separate from the repeated benchmark.


In [ ]:
# Use one fixed protocol case per true class, independently of model predictions.
representatives = [
    (class_id, next(index for index in explanation_cases.index if int(y_test.loc[index]) == class_id))
    for class_id in range(len(class_names))
]
fig, axes = plt.subplots(2, 4, figsize=(22, 10), squeeze=False)
for ax, (class_id, case_id) in zip(axes.flat, representatives):
    row = X_test.loc[case_id]
    predicted_label = int(model.predict(X_test.loc[[case_id]])[0])
    label_index = int(np.flatnonzero(model.classes_ == predicted_label)[0])
    # A fresh seed gives each illustrative case the protocol's starting random state.
    case_explainer = LimeTabularExplainer(
        lime_background.to_numpy(), feature_names=feature_names, class_names=class_names,
        categorical_features=categorical_feature_indices,
        categorical_names={i: ["0", "1"] for i in categorical_feature_indices},
        mode="classification", random_state=RANDOM_STATE,
    )
    exp = case_explainer.explain_instance(
        row.to_numpy(), predict_proba_for_lime, labels=(label_index,),
        num_features=NUM_FEATURES, num_samples=SAMPLE_BUDGETS[-2],
    )
    terms = pd.DataFrame(exp.as_list(label=label_index), columns=["condition", "weight"]).sort_values("weight")
    ax.barh(terms.condition, terms.weight,
        color=["#4C78A8" if weight < 0 else "#F58518" for weight in terms.weight])
    ax.axvline(0, color="black", linewidth=.8)
    ax.set_title(f"Actual {class_names[class_id]} | predicted {class_names[predicted_label]}", fontsize=9)
    ax.tick_params(axis="y", labelsize=7)
for ax in axes.flat[len(representatives):]:
    ax.axis("off")
fig.suptitle("Illustrative explanations for fixed shared cases")
plt.tight_layout()
plt.show()


## Controlled LIME comparison

`controlled_lime_protocol.py` is the source of shared experiment settings. Every model receives the same stratified 10,000-row sample, 8,000 training rows, 2,000 held-out rows, 54 input predictors, and labels 0–6. The protocol chooses three held-out cases per true class without filtering by predictions, then uses the same 2,000-row training background, budgets, five repeat seeds (42–46), ten explanation terms, and stability definitions.

The 18/36/54-feature subsets change which predictors LIME perturbs. Models are fitted once on all 54 predictors; excluded predictors remain fixed at each case's value. Runtime includes LIME's prediction calls and surrogate fitting but excludes explainer setup and prediction warm-up. Fidelity is the local surrogate's weighted R², not predictive accuracy. Stability uses signed-weight Spearman on selected-feature unions and top-five Jaccard.

Model architecture, its hyperparameters and required preprocessing remain model-specific. Compare timing on the same hardware and backend where possible; CUDA and CPU timings have different resource conditions. Results below must be regenerated after changing the protocol.


In [ ]:
# The shared function owns sampling, seeds, feature subsets and stability calculations.
model_name = 'XGBoost'
controlled_raw, controlled_summary = benchmark_lime(
    model, X_train, X_test, y_test, predict_proba_for_lime, model_name
)
predictive_metrics = evaluate_predictive_performance(model, X_test, y_test)
print("Held-out predictive performance:", predictive_metrics)
display(controlled_summary.round(4))

# Write beside the protocol, regardless of whether Jupyter started at the repo root.
import controlled_lime_protocol as protocol
output_directory = Path(protocol.__file__).resolve().parent
controlled_raw.to_csv(output_directory / f"{model_name.lower()}_controlled_lime_raw.csv", index=False)
controlled_summary.to_csv(output_directory / f"{model_name.lower()}_controlled_lime_summary.csv", index=False)
pd.DataFrame([{"model": model_name, **predictive_metrics}]).to_csv(
    output_directory / f"{model_name.lower()}_controlled_predictive_metrics.csv", index=False
)


### Reading the controlled results

Compare matching `feature_count` and `num_samples` rows between models. Lower runtime and higher fidelity/stability are preferable. The dimensionality experiment changes LIME's perturbations, while model training always uses all 54 input features.


## Dataset, objective, and shared comparison method

Covertype contains 581,012 forest cells, each representing a 30 × 30 metre area. This notebook uses the protocol's stratified 10,000-row sample with 54 terrain, distance, hillshade, wilderness, and soil features to predict one of seven cover types.

The controlled benchmark uses the same seeded train/test split, held-out cases, training-background sample, perturbation budgets, repeat seeds, and nested feature subsets as the other model notebooks. It evaluates perturbation dimensions of 18, 36, and 54 features; excluded features stay fixed at each case's observed value. This measures LIME explanation-space scaling, not model retraining with fewer predictors. XGBoost's architecture and fitted model remain XGBoost-specific.


## Controlled results across budgets and perturbation dimensions


In [17]:
# Plot the shared summary separately for each perturbation dimension.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for feature_count, group in controlled_summary.groupby("feature_count"):
    group = group.sort_values("num_samples")
    label = f"{feature_count} perturbed features"
    axes[0, 0].errorbar(group["num_samples"], group["runtime_mean_seconds"],
        yerr=group["runtime_std_seconds"], marker="o", capsize=3, label=label)
    axes[0, 1].errorbar(group["num_samples"], group["fidelity_mean"],
        yerr=group["fidelity_std"], marker="o", capsize=3, label=label)
    axes[1, 0].plot(group["num_samples"], group["spearman_stability_mean"], marker="o", label=label)
    axes[1, 1].plot(group["num_samples"], group["top5_jaccard_mean"], marker="o", label=label)

for ax, title, ylabel in zip(axes.flat,
    ["Explanation runtime", "Local fidelity", "Signed-weight rank stability", "Top-five feature stability"],
    ["Seconds (mean ± SD)", "Weighted surrogate R² (mean ± SD)", "Mean pairwise Spearman", "Mean pairwise Jaccard"]):
    ax.set(title=title, xlabel="Perturbation samples", ylabel=ylabel, xscale="log")
    ax.set_xticks(SAMPLE_BUDGETS, labels=[f"{budget:,}" for budget in SAMPLE_BUDGETS])
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0, 0].set_ylim(bottom=0)
axes[1, 0].set_ylim(-1.05, 1.05)
axes[1, 1].set_ylim(-0.05, 1.05)
fig.suptitle(f"{model_name}: controlled LIME comparison")
plt.tight_layout()
plt.show()


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3579, in run_code
  File "/var/folders/z9/hp16j7rj42g713h_xczg2g4m0000gq/T/ipykernel_25170/2903229107.py", line 19, in <module>
    ax.set_xticks(SAMPLE_BUDGETS, labels=[f"{budget:,}" for budget in SAMPLE_BUDGETS])
NameError: name 'SAMPLE_BUDGETS' is not defined. Did you mean: 'LIME_SAMPLE_BUDGETS'?

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/pygments/styles/__init__.py", line 45, in get_style_by_name
ModuleNotFoundError: No module named 'pygments.styles.default'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/ExtraAccount/Documents/college subject

Error in callback <function _draw_all_if_interactive at 0x11b3fb5b0> (for post_execute), with arguments args (),kwargs {}:
Unexpected exception formatting exception. Falling back to standard exception
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/IPython/core/formatters.py", line 282, in catch_format_error
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/IPython/core/formatters.py", line 402, in __call__
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/IPython/core/pylabtools.py", line 170, in print_figure
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/matplotlib/backend_bases.py", line 2157, in print_figure
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/python3.10/site-packages/matplotlib/artist.py", line 94, in draw_wrapper
  File "/Users/ExtraAccount/Documents/college subjects/block_10/xAI/xAI_Project/XAI/.venv/lib/pyth

<Figure size 1300x800 with 4 Axes>